# The data

This is the first in a short series of notebooks introducing `matched`: how the input data is shaped, how it is cleaned, and how the allocation algorithm itself works. We start with the data supervisors and students provide.

### `projects.csv`

In [ ]:
import pathlib

import pandas as pd

raw = pathlib.Path("./raw")

projects = pd.read_csv(raw / "projects.csv").set_index("code")

projects.head(3)

`projects.csv` contains data collected from supervisors. Each project is given a unique code (`code (str)`). For each project code, we record:
- `external (bool)` indicating whether the project is external or internal
- `nmax (int>0)` - the maximum number of students that can be assigned to the project
- `course_i (bool)` - which course(s) the project is available for.

From the data above, we can see that the project `mabe-007` is an internal project that can be assigned to at most 3 students, and is available to `course1` only.

### `internal-choices.csv`

In [ ]:
choices_wide = pd.read_csv(raw / "internal-choices.csv").set_index("username")

choices_wide.head(3)

Each student fills in a form where they select $N$ projects they are interested in, in order of preference - one column per rank (`choice1`, `choice2`, ...). For each student, we also record which `course` they are enrolled on, and their `score` (e.g. academic performance), used as a tiebreaker when multiple students want the same project.

### Reshaping into the `choices` format

`matched` expects one row *per student preference* - `username`, `code`, `choice`, `score` (and optionally `course`) - rather than one row per student with several choice columns. We reshape the wide form above into that long, tidy format:

In [ ]:
choice_cols = [c for c in choices_wide.columns if c.startswith("choice")]

choices = choices_wide.reset_index().melt(
    id_vars=["username", "course", "score"],
    value_vars=choice_cols,
    var_name="choice",
    value_name="code",
)
choices["choice"] = choices["choice"].str.removeprefix("choice").astype(int)
choices = choices.sort_values(["username", "choice"]).reset_index(drop=True)

choices.head(8)

Each student now has one row per project they picked, ranked by `choice` (1 = first preference). This is the shape `matched.preprocess` and `matched.match` expect. In [Preprocessing](02-preprocessing.ipynb), we clean this data before allocating.